In [1]:
import pandas as pd
import numpy as np
import os
import pandas_gbq
from google.cloud import bigquery
import glob
import openpyxl

c:\Users\ana.sales_republica\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
os.getcwd()

'g:\\Drives compartilhados\\República.org\\02. Áreas\\Dados e Conhecimento\\415 - Repositório de Dados\\Tratamento Github\\Ecossistema-dados\\tratamento_GBQ\\cargos_lideranca'

Chamando as bases

In [ ]:
dfseg19 = pd.read_excel('munic_2019.xlsx', sheet_name='Segurança pública', usecols=['CodMun', 'COD UF','NOME MUNIC','MSEG01','MSEG03', 'MSEG04', 'MSEG05', 'MSEG06'])
dfseg19['ano']= 2019
uf = pd.read_excel('munic_2019.xlsx', sheet_name = 'Variáveis externas', usecols=[1,3]) # Pegando nome e codigo das UF
dfseg19 = dfseg19.rename(columns={'COD UF':'cod_uf',
                                'CodMun':'cod_municipio',
                                'NOME MUNIC':'nome_municipio', 
                                'MSEG01':'secretaria',
                                'MSEG03':'genero',
                                'MSEG04':'idade',
                                'MSEG05':'cor_raca',
                                'MSEG06':'grau_instrucao'
                             })


In [ ]:
mseg19[mseg19['genero']=='Sem titular'] #Unico com secretaria que sem informação

In [ ]:
dfseg19['cor_raca'] = np.where(dfseg19['cor_raca'] == 'Pardo', 'Parda', dfseg19['cor_raca'])

# Handle missing/refused data for gender column
dfseg19['genero'] = np.where(dfseg19['genero'] == 'Recusa', 'Sem dados', dfseg19['genero'])
dfseg19['genero'] = np.where(dfseg19['genero'] == 'Não informou', 'Sem dados', dfseg19['genero'])
dfseg19['genero'] = np.where(dfseg19['genero'] == '-', 'Sem dados', dfseg19['genero'])
dfseg19['genero'] = np.where(dfseg19['genero'] == '(**) Sem gestor', 'Sem dados', dfseg19['genero'])
dfseg19['genero'] = np.where(dfseg19['genero'] == 'Não soube informar', 'Sem dados', dfseg19['genero'])

# Handle missing/refused data for race/color column
dfseg19['cor_raca'] = np.where(dfseg19['cor_raca'] == 'Recusa', 'Sem dados', dfseg19['cor_raca'])
dfseg19['cor_raca'] = np.where(dfseg19['cor_raca'] == 'Não informou', 'Sem dados', dfseg19['cor_raca'])
dfseg19['cor_raca'] = np.where(dfseg19['cor_raca'] == '-', 'Sem dados', dfseg19['cor_raca'])
dfseg19['cor_raca'] = np.where(dfseg19['cor_raca'] == '(**) Sem gestor', 'Sem dados', dfseg19['cor_raca'])
dfseg19['cor_raca'] = np.where(dfseg19['cor_raca'] == 'Não soube informar', 'Sem dados', dfseg19['cor_raca'])

# Handle missing/refused data for education level column
dfseg19['grau_instrucao'] = np.where(dfseg19['grau_instrucao'] == 'Recusa', 'Sem dados', dfseg19['grau_instrucao'])
dfseg19['grau_instrucao'] = np.where(dfseg19['grau_instrucao'] == 'Não informou', 'Sem dados', dfseg19['grau_instrucao'])
dfseg19['grau_instrucao'] = np.where(dfseg19['grau_instrucao'] == '-', 'Sem dados', dfseg19['grau_instrucao'])
dfseg19['grau_instrucao'] = np.where(dfseg19['grau_instrucao'] == '(**) Sem gestor', 'Sem dados', dfseg19['grau_instrucao'])
dfseg19['grau_instrucao'] = np.where(dfseg19['grau_instrucao'] == 'Não soube informar', 'Sem dados', dfseg19['grau_instrucao'])

# Handle missing/refused data for age column (convert to NaN)
dfseg19['idade'] = np.where(dfseg19['idade'] == 'Recusa', np.nan, dfseg19['idade'])
dfseg19['idade'] = np.where(dfseg19['idade'] == 'Não informou', np.nan, dfseg19['idade'])
dfseg19['idade'] = np.where(dfseg19['idade'] == '-', np.nan, dfseg19['idade'])
dfseg19['idade'] = np.where(dfseg19['idade'] == '(**) Sem gestor', np.nan, dfseg19['idade'])
dfseg19['idade'] = np.where(dfseg19['idade'] == 'Não soube informar', np.nan, dfseg19['idade'])

# Convert age column to numeric type
dfseg19['idade'] = pd.to_numeric(dfseg19['idade'])

# Define age group bins and labels
limites = [18, 30, 50, 65, 100]
categorias = ['Entre 18-29', 'Entre 30-49', 'Entre 50-64', 'Acima de 65']

# Create age group column based on the bins
dfseg19['faixa_etaria'] = pd.cut(dfseg19['idade'], bins=limites, labels=categorias, right=False)

# Reorder columns in the dataframe
dfseg19 = dfseg19[['ano', 'sigla_uf', 'nome_municipio', 'id_municipio', 'genero', 'cor_raca', 'grau_instrucao', 'faixa_etaria']]

# Create dictionary to standardize education levels
dict_esco = {
    'Ensino fundamental incompleto': 'Até Ensino Fundamental',
    'Ensino fundamental completo': 'Até Ensino Fundamental',
    'Ensino fundamental (1º Grau) completo': 'Até Ensino Fundamental',
    'Ensino fundamental (1º Grau) incompleto': 'Até Ensino Fundamental',
    'Ensino médio (2º Grau) incompleto': 'Até Ensino Médio',
    'Ensino médio (2º Grau) completo': 'Até Ensino Médio',
    'Ensino superior incompleto': 'Até Ensino Superior Completo',
    'Ensino superior completo': 'Até Ensino Superior Completo',
    'Especialização': 'Até Pós Graduação ou Mestrado',
    'Mestrado': 'Até Pós Graduação ou Mestrado',
    'Doutorado': 'Até Doutorado'
}

# Apply education level standardization
dfseg19 = dfseg19.replace({'grau_instrucao': dict_esco})

# Display unique values in education column for verification
dfseg19['grau_instrucao'].unique()

### 2023

In [ ]:
dfseg23 = pd.read_excel('munic_2023.xlsx', sheet_name='Segurança Pública', usecols=['CodMun', 'Cod UF','Mun','MSEG01','MSEG03', 'MSEG04', 'MSEG05', 'MSEG06'])
dfseg23['ano']= 2023
dfseg23 = dfseg23.rename(columns={'Cod UF':'cod_uf',
                                'MSEG01':'secretaria',
                                'CodMun':'cod_municipio',
                                'Mun':'nome_municipio', 
                                'MSEG03':'genero',
                                'MSEG04':'idade',
                                'MSEG05':'cor_raca',
                                'MSEG06':'grau_instrucao'
                             })
dfseg23['nome_municipio'] = dfseg23['nome_municipio'].str.title()
dfseg23 = dfseg23[['ano','cod_uf','id_municipio','nome_municipio','caracterizacao_orgao_gestor','genero','idade','cor_raca','grau_instrucao']]
cols = ['caracterizacao_orgao_gestor','genero', 'grau_instrucao','cor_raca']
for col in cols:
    dfseg23[col]= dfseg23[col].str.replace('(**) Sem gestor','Sem dados')
    dfseg23[col]= dfseg23[col].str.replace('Não informou','Sem dados')
    dfseg23[col]= dfseg23[col].str.replace('-','Sem dados')
dfseg23['idade']=np.where(dfseg23['idade']=='-',np.nan,dfseg23['idade']) 
dfseg23['idade']=np.where(dfseg23['idade']=='Não informou',np.nan,dfseg23['idade']) 
dfseg23['idade']=np.where(dfseg23['idade']=='(**) Sem gestor',np.nan,dfseg23['idade'])  
dfseg23['idade']=np.where(dfseg23['idade']=='-',np.nan,dfseg23['idade'])  
dfseg23 

In [ ]:
dfseg23['cor_raca'] = np.where(dfseg23['cor_raca'] == 'Pardo', 'Parda', dfseg23['cor_raca'])

# Handle missing/refused data for gender column
dfseg23['genero'] = np.where(dfseg23['genero'] == 'Recusa', 'Sem dados', dfseg23['genero'])
dfseg23['genero'] = np.where(dfseg23['genero'] == 'Não informou', 'Sem dados', dfseg23['genero'])
dfseg23['genero'] = np.where(dfseg23['genero'] == '-', 'Sem dados', dfseg23['genero'])
dfseg23['genero'] = np.where(dfseg23['genero'] == '(**) Sem gestor', 'Sem dados', dfseg23['genero'])
dfseg23['genero'] = np.where(dfseg23['genero'] == 'Não soube informar', 'Sem dados', dfseg23['genero'])

# Handle missing/refused data for race/color column
dfseg23['cor_raca'] = np.where(dfseg23['cor_raca'] == 'Recusa', 'Sem dados', dfseg23['cor_raca'])
dfseg23['cor_raca'] = np.where(dfseg23['cor_raca'] == 'Não informou', 'Sem dados', dfseg23['cor_raca'])
dfseg23['cor_raca'] = np.where(dfseg23['cor_raca'] == '-', 'Sem dados', dfseg23['cor_raca'])
dfseg23['cor_raca'] = np.where(dfseg23['cor_raca'] == '(**) Sem gestor', 'Sem dados', dfseg23['cor_raca'])
dfseg23['cor_raca'] = np.where(dfseg23['cor_raca'] == 'Não soube informar', 'Sem dados', dfseg23['cor_raca'])

# Handle missing/refused data for education level column
dfseg23['grau_instrucao'] = np.where(dfseg23['grau_instrucao'] == 'Recusa', 'Sem dados', dfseg23['grau_instrucao'])
dfseg23['grau_instrucao'] = np.where(dfseg23['grau_instrucao'] == 'Não informou', 'Sem dados', dfseg23['grau_instrucao'])
dfseg23['grau_instrucao'] = np.where(dfseg23['grau_instrucao'] == '-', 'Sem dados', dfseg23['grau_instrucao'])
dfseg23['grau_instrucao'] = np.where(dfseg23['grau_instrucao'] == '(**) Sem gestor', 'Sem dados', dfseg23['grau_instrucao'])
dfseg23['grau_instrucao'] = np.where(dfseg23['grau_instrucao'] == 'Não soube informar', 'Sem dados', dfseg23['grau_instrucao'])

# Handle missing/refused data for age column (convert to NaN)
dfseg23['idade'] = np.where(dfseg23['idade'] == 'Recusa', np.nan, dfseg23['idade'])
dfseg23['idade'] = np.where(dfseg23['idade'] == 'Não informou', np.nan, dfseg23['idade'])
dfseg23['idade'] = np.where(dfseg23['idade'] == '-', np.nan, dfseg23['idade'])
dfseg23['idade'] = np.where(dfseg23['idade'] == '(**) Sem gestor', np.nan, dfseg23['idade'])
dfseg23['idade'] = np.where(dfseg23['idade'] == 'Não soube informar', np.nan, dfseg23['idade'])

# Convert age column to numeric type
dfseg23['idade'] = pd.to_numeric(dfseg23['idade'])

# Define age group bins and labels
limites = [18, 30, 50, 65, 100]
categorias = ['Entre 18-29', 'Entre 30-49', 'Entre 50-64', 'Acima de 65']

# Create age group column based on the bins
dfseg23['faixa_etaria'] = pd.cut(dfseg23['idade'], bins=limites, labels=categorias, right=False)

# Reorder columns in the dataframe
dfseg23 = dfseg23[['ano', 'sigla_uf', 'nome_municipio', 'id_municipio', 'genero', 'cor_raca', 'grau_instrucao', 'faixa_etaria']]

# Create dictionary to standardize education levels
dict_esco = {
    'Ensino fundamental incompleto': 'Até Ensino Fundamental',
    'Ensino fundamental completo': 'Até Ensino Fundamental',
    'Ensino fundamental (1º Grau) completo': 'Até Ensino Fundamental',
    'Ensino fundamental (1º Grau) incompleto': 'Até Ensino Fundamental',
    'Ensino médio (2º Grau) incompleto': 'Até Ensino Médio',
    'Ensino médio (2º Grau) completo': 'Até Ensino Médio',
    'Ensino superior incompleto': 'Até Ensino Superior Completo',
    'Ensino superior completo': 'Até Ensino Superior Completo',
    'Especialização': 'Até Pós Graduação ou Mestrado',
    'Mestrado': 'Até Pós Graduação ou Mestrado',
    'Doutorado': 'Até Doutorado'
}

# Apply education level standardization
dfseg23 = dfseg23.replace({'grau_instrucao': dict_esco})

# Display unique values in education column for verification
dfseg23['grau_instrucao'].unique()

In [ ]:
dfseg = pd.concat([dfseg19,dfseg23], ignore_index=True)
dfseg

In [146]:
# criando dicionário
dict_esco = {'Sem dados':'Sem dados',
            'Especialização':'Até Pós Graduação ou Mestrado',
            'Ensino superior completo':'Até Ensino Superior Completo',
            'Ensino superior incompleto':'Até Ensino Superior Completo',
            'Ensino fundamental (1º Grau) incompleto':'Até Ensino Médio',
            'Ensino médio (2º Grau) completo':'Até Ensino Médio',
            'Ensino médio (2º Grau) incompleto':'Até Ensino Médio',
            'Mestrado':'Até Pós Graduação ou Mestrado',
            'Ensino fundamental (1º Grau) completo':'Até Ensino Médio',
            'Doutorado':'Até Doutorado',
            'Ensino fundamental ( 1º Grau) completo':'Até Ensino Médio'}


In [ ]:
dfseg['grau_instrucao'].unique()

array(['Sem dados', 'Especialização', 'Ensino superior completo',
       'Ensino superior incompleto',
       'Ensino fundamental (1º Grau) incompleto',
       'Ensino médio (2º Grau) completo',
       'Ensino médio (2º Grau) incompleto', 'Mestrado',
       'Ensino fundamental (1º Grau) completo', 'Doutorado',
       'Ensino fundamental ( 1º Grau) completo'], dtype=object)

In [ ]:
dfseg = dfseg.replace({'grau_instrucao':dict_esco}) #substituindo valores para padronizar
dfseg

In [ ]:
dfseg['idade'] =pd.to_numeric(dfseg['idade'])

In [ ]:
limites = [0, 29, 49, 64, 100] #criando uma nova coluna (faixa_etaria) com base na coluna 'idade'
categorias = ['Entre 18-29', 'Entre 30-49', 'Entre 50-65', 'Acima de 65']

dfseg['faixa_etaria'] = pd.cut(dfseg['idade'], bins=limites, labels=categorias)


In [ ]:
dfseg[dfseg['idade']==49]['faixa_etaria'].value_counts()

Subindo para o GBQ

In [158]:
client = bigquery.Client()
dataset_ref = client.dataset('cargos_lideranca')

In [ ]:
dfseg.info()

In [ ]:
dfseg = dfseg[['ano', 'cod_uf','id_municipio','nome_municipio','caracterizacao_orgao_gestor','genero', 'faixa_etaria', 'cor_raca', 'grau_instrucao']]
dfseg

In [161]:
schema=[bigquery.SchemaField('ano','INTEGER',description='Ano referente a informação'),
        bigquery.SchemaField('cod_uf','INTEGER',description='Sigla da UF'),
        bigquery.SchemaField('id_municipio','INTEGER',description='Código do IBGE da UF'),
        bigquery.SchemaField('nome_municipio','STRING',description='Caracterização do órgão no qual o gestor está'), 
        bigquery.SchemaField('caracterizacao_orgao_gestor','STRING',description='Nome da UF'), 
        bigquery.SchemaField('genero','STRING',description='Gênero autodeclarado ou não'),
        bigquery.SchemaField('faixa_etaria','STRING',description='Faixa etária da observação'),
        bigquery.SchemaField('cor_raca','STRING',description='Raça/cor da pessoa observada'),
        bigquery.SchemaField('grau_instrucao','STRING',description='Escolaridade da pessoa ou do vínculo observado com detalhamento na pós-graduação')
        ]


In [ ]:
table_ref = dataset_ref.table('MUNIC_perfil_gestor_seguranca_publica_tipo_orgao')
job_config = bigquery.LoadJobConfig(schema=schema)
job = client.load_table_from_dataframe(dfseg,table_ref, job_config=job_config)
job.result() 

LoadJob<project=repositoriodedadosgpsp, location=US, id=579efad5-14b7-48a9-afa2-ae44849a8f24>